# Does the gate pick better frames for a VLM? EgoEverything, 12 videos

[EgoEverything](https://huggingface.co/datasets/roxqtang/EgoEverything) (arXiv 2604.08342)
asks 5-option questions about what an AR-glasses wearer saw, built from real gaze traces on
Aria Everyday Activities (AEA). Each question names a **keyframe time** where the answer is
visible. 834 questions fall on 12 of the gate's videos: 4 test videos, and 8 training videos
scored by the fold model that held each one out.

**The test:** at the **same cost**, does the gate's choice of frames give the VLM better
answers than a **fixed interval chosen in advance**?

## Findings so far (Qwen2.5-VL-3B, 448 px)

| condition | frames / min | accuracy |
|---|---|---|
| fixed every 6 s | 10.1 | 42.3% |
| policy D (send when the view settles + on each REFIXATION) | 10.6 | 40.5% |
| fixed every 5 s | 12.2 | 42.5% |

D was ~1.8 points below a fixed interval at equal cost (not significant). D left long gaps:
too few frames on quiet videos (7 frames in 107 s on one) and none during long motion.

## This run: Full against content-based selectors on the 1 fps stream

All methods see one candidate frame per second and decide dynamically:

| method | decides from |
|---|---|
| **Full** (policy D, eye + IMU model) | gaze + IMU, no pixels |
| 1 fps + DINOv2 change | DINOv2 CLS of every frame vs the last sent one (τ_frame) |
| 1 fps + pixel change | grey thumbnail of every frame vs the last sent one |
| 1 fps: all frames | sends everything (the ceiling) |

Reported per method: **accuracy, frames sent to the VLM, visual tokens, and the FLOPs of the
send/skip decision** (plus approximate VLM compute). Section 6c shows frames, coverage and
decision cost before any GPU time; section 9 adds accuracy.

## Next: temporal + spatial (section 6d, no GPU)

Full decides *which* frames to send; the spatial module (`apt-vlm`, gaze-foveated patches)
decides *how many tokens* each part of a sent frame gets. Section 6d checks, without the VLM,
whether gaze puts the fine patches on the questions' targets (against centre and
other-time gaze), whether the combination keeps the target in view, and whether D sends at
moments when the gaze is a fixation.

## Run order

1. Clone, mount, install  2. AEA links JSON  3. Questions  4. Held-out predictions and
frame selection  5. Frames (cached on Drive)  6. Alignment and coverage  6b. Gap screen
6c. Baselines (no GPU)  **6d. Spatial screen (no GPU)**  7. Load Qwen  8. Score  9. Results

Earlier results are in the same resumable file and reused.

## 1. Clone, mount, install

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('vlmeval/qwen_mcq.py'), f'vlmeval/ is not on branch {BRANCH!r}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'qwen-vl-utils', 'huggingface_hub'],
               check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
VLM_DIR   = os.path.join(DRIVE_DIR, 'vlm', 'egoeverything')
FRAMES_TAR = os.path.join(VLM_DIR, 'frames_tar')          # one .tar per video, reused later
LOCAL_FRAMES = '/content/frames_1fps'
os.makedirs(VLM_DIR, exist_ok=True)

CKPT_FULL = os.path.join(RUNS, 'eyehead')                  # the full model's 5 fold checkpoints
missing = [i for i in range(1, 6) if not os.path.exists(os.path.join(CKPT_FULL, f'fold{i}.pt'))]
assert not missing, f'missing fold checkpoints {missing} in {CKPT_FULL} -- run the eyehead notebook first'
print('full model checkpoints: 5 folds found in', CKPT_FULL)
FEAT_CACHE = os.path.join(DRIVE_DIR, 'feat_cache.npz')     # DINOv2 CLS per frame (written by gatetrain)
print('DINOv2 feature cache:', 'found' if os.path.exists(FEAT_CACHE) else 'MISSING (needed for the DINOv2 baseline)')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. The AEA download-links JSON

The signed links expire after ~14 days. This cell checks the copy on Drive:

- **missing or expired** → it asks you to upload the fresh `aea_download_urls.json`, checks
  its links work, and **replaces the older copies** on Drive (`MyDrive/GazeVLM/` and, if
  present, `MyDrive/aea/`, so every notebook picks up the fresh one)
- **still valid** → nothing to do

Set `FORCE_UPLOAD = True` to replace it anyway.

In [ ]:
from vlmeval.frames import links_fresh, install_links

FORCE_UPLOAD = False
LINKS = os.path.join(DRIVE_DIR, 'aea_download_urls.json')
OLD_COPIES = [LINKS] + [p for p in ['/content/drive/MyDrive/aea/aea_download_urls.json']
                        if os.path.exists(p)]

ok, msg = links_fresh(LINKS) if os.path.exists(LINKS) else (False, 'not on Drive')
print('Drive copy:', msg)
if FORCE_UPLOAD or not ok:
    from google.colab import files
    print('upload the fresh aea_download_urls.json ...')
    up = files.upload()
    name = list(up.keys())[0]
    ok, msg = links_fresh(name)
    print('uploaded file:', msg)
    assert ok, 'the uploaded JSON does not work either -- download a fresh one from projectaria.com/datasets/aea'
    install_links(name, OLD_COPIES)
    os.remove(name)                      # the copies live on Drive; keep the repo clean
    for p in OLD_COPIES:
        print('   replaced', p)
links = json.load(open(LINKS))
print(f"using {LINKS}: {len(links['sequences'])} sequences")
print('(the eyehead notebook\'s IMU calibration, section 3, can now download its VRS too)')

## 3. EgoEverything questions on the gate's videos

In [ ]:
from vlmeval import egoev

qs_all = egoev.load_questions()
aea = qs_all[qs_all.video.str.startswith('loc')]
clips_all, test = egoev.gate_clips(FOLDS, test_only=False)
gate_videos = {c['sequence'] for c in clips_all}
test_videos = {c['sequence'] for c in clips_all if c['is_test']}

qs = aea[aea.video.isin(gate_videos)].reset_index(drop=True)
clips = [c for c in clips_all if c['sequence'] in set(qs.video)]
videos = [c['sequence'] for c in clips]
n_frames = {c['sequence']: len(c['frame_no']) for c in clips}
print(f'EgoEverything: {len(qs_all):,} questions, {len(aea):,} on AEA ({aea.video.nunique()} videos)')
print(f'on the gate videos: {len(qs):,} questions on {len(videos)} videos '
      f'({qs.video.isin(test_videos).sum()} questions on {len(set(videos) & test_videos)} test videos)\n')

per = qs.groupby('video').agg(questions=('qid', 'size'), t0_max=('t0', 'max'),
                              median_dist2gaze=('dist2gaze', 'median'))
per['video_s'] = [n_frames[v] for v in per.index]
per['split'] = ['test' if v in test_videos else 'train (held out by its fold)' for v in per.index]
display(per.sort_values('questions', ascending=False).round(1))
print('answer letters:', qs.answer.value_counts().sort_index().to_dict())

## 4. Held-out gate predictions, and which frames each condition sends

Each training video is scored by the fold model that held it out for validation; test
videos by the 5-fold ensemble. The table is cached on Drive. The quick accuracy check below
should be close to the cross-validation numbers (~0.6 four-class accuracy), not higher,
which would mean a video leaked into its own training.

In [ ]:
from eyehead.oof import oof_predictions

OOF = os.path.join(VLM_DIR, 'oof_preds_full.csv')
if os.path.exists(OOF):
    oof = pd.read_csv(OOF)
    print('loaded', OOF)
else:
    oof = oof_predictions(FOLDS, CKPT_FULL)
    oof.to_csv(OOF, index=False)
    print('wrote', OOF)

o12 = oof[oof.sequence.isin(videos)]
print(f"\nheld-out 4-class accuracy on these {len(videos)} videos: "
      f"{(o12.pred == o12.quad).mean():.3f}   by source:")
display(o12.assign(correct=o12.pred == o12.quad).groupby('source').correct.agg(['mean', 'size']).round(3))

labels = {'full': egoev.video_labels(clips, oof)}
sel = egoev.select_conditions(clips, labels, main='full', timeout=5, hybrid_frac=0.5, seed=0)
json.dump(sel, open(os.path.join(VLM_DIR, 'selection_all12.json'), 'w'))
display(egoev.budget_table(sel, n_frames).round(2))

# per video: frames each condition sends; D_interval_s = the spacing the per-video-matched
# uniform used, which a deployed fixed interval cannot adapt to
pv_budget = pd.DataFrame({c: {v: len(sel[c][v]) for v in videos}
                          for c in ('D: full', 'fixed every 5 s', 'fixed every 6 s')})
pv_budget.insert(0, 'video_s', [n_frames[v] for v in pv_budget.index])
pv_budget['D_interval_s'] = (pv_budget['video_s'] / pv_budget['D: full']).round(1)
display(pv_budget.sort_values('D_interval_s'))
print(f"D: full sends one frame every {pv_budget.video_s.sum() / pv_budget['D: full'].sum():.1f} s on average")

## 5. Frames: download once, cache on Drive

The 4 test videos are already cached from the pilot; the 8 others are downloaded once
(~165 MB each).

In [ ]:
from vlmeval.frames import ensure_frames, frame_path

SIZE = 448
metas = {}
for v in videos:
    metas[v] = ensure_frames(v, links, LOCAL_FRAMES, FRAMES_TAR, size=SIZE)
    m = metas[v]
    flag = 'OK' if m['n'] >= n_frames[v] else f'!! only {m["n"]} frames, labels need {n_frames[v]}'
    print(f"   {v:<26} {m['n']:4d} frames  {flag}")

## 6. Alignment check and target coverage

**Alignment:** the frame nearest each question's keyframe, with its box. The boxes should sit
on the named objects.

**Target coverage:** distance from each question's keyframe to the nearest sent frame. The
pilot showed this decides the score: D covered 40% of keyframes within 2 s against 66% for
fixed interval. The hybrid should close most of that gap.

In [ ]:
import matplotlib.patches as mpatches

def frame_time(v, k):
    return metas[v]['ts'][k] if k < metas[v]['n'] else k

show = qs[qs.bbox.notna() & qs.t0.notna()].sample(min(6, len(qs)), random_state=1)
fig, ax = plt.subplots(1, len(show), figsize=(3.4 * len(show), 3.8))
for a_, q in zip(np.atleast_1d(ax), show.itertuples()):
    m = metas[q.video]
    k = int(np.argmin(np.abs(np.array(m['ts']) - q.t0)))
    img = plt.imread(frame_path(m, k))
    s = img.shape[1] / m['orig_w']
    x1, y1, x2, y2 = [b * s for b in q.bbox]
    a_.imshow(img)
    a_.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='yellow', lw=2))
    a_.set_title(f"{q.object}\nt0={q.t0:.1f}s  frame {k}", fontsize=8); a_.axis('off')
plt.tight_layout(); plt.show()

cov = egoev.coverage(qs, sel, frame_time)
ctab = cov.groupby('condition', sort=False).agg(
    questions=('qid', 'size'), median_dist_s=('dist', 'median'),
    **{f'within_{t}': (f'hit_{t}', 'mean') for t in ('1s', '2s', '5s')})
for c in ('within_1s', 'within_2s', 'within_5s'):
    ctab[c] *= 100
display(ctab.round(1))

## 6b. Screen the gap-capped policy (no GPU)

**Policy "D + max gap G s":** send exactly what policy D sends (when the view settles, and
on each REFIXATION), but also send a frame whenever G seconds have passed since the last
one. It is causal, so it is deployable. The cap targets D's main failure in the last run:
on quiet videos D sent too little (e.g. 7 frames in 107 s on `loc2_script2_seq2`).

Each G is compared with a **fixed interval at the same total frame rate**, chosen as one
number for all videos (deployable). Coverage = how often a frame lands within 1 s / 2 s of
the moment a question is about; in the last run, coverage decided accuracy.

`BEST_GAP` is the G with the largest 2 s coverage advantage over its fixed interval.
**Only run the VLM (section 8) if that advantage is clearly positive;** otherwise the gap
cap cannot beat a fixed interval on these questions and the GPU time is better spent
elsewhere.

In [ ]:
GAPS = (5, 6, 8, 10)
scr = egoev.gap_screen(sel, cov, GAPS, main='full', n_frames=n_frames)
print('gap-capped D vs a fixed interval with the same total frame rate (coverage, % of 834 questions):')
display(scr.round(1))

gt = pd.DataFrame({g: {
        'D + max gap (DINOv2 labels) within_2s': ctab.loc[f'D + max gap {g} s: ground truth', 'within_2s'],
        'D + max gap (full model) within_2s':    ctab.loc[f'D + max gap {g} s: full', 'within_2s']}
    for g in GAPS}).T.rename_axis('max_gap_s')
print('same policy on the DINOv2 labels (its ceiling) vs the model:')
display(gt.round(1))

BEST_GAP = int(scr['advantage_2s'].idxmax())
adv = scr.loc[BEST_GAP, 'advantage_2s']
print(f"\nBEST_GAP = {BEST_GAP} s: {scr.loc[BEST_GAP, 'frames_per_min']:.1f} frames/min, "
      f"2 s coverage {scr.loc[BEST_GAP, 'policy_within_2s']:.1f}% vs fixed "
      f"{scr.loc[BEST_GAP, 'fixed_within_2s']:.1f}%  (advantage {adv:+.1f} points)")
print('-> worth a VLM run' if adv > 2 else
      '-> no clear coverage advantage: a VLM run is unlikely to beat the fixed interval')

## 6c. Baselines on the same 1 fps stream (no GPU)

Every method sees one candidate frame per second and decides per frame whether to send it.
Each decides dynamically, by its own rule:

| method | decides from | sends a frame when |
|---|---|---|
| **Full** (policy D, eye + IMU model) | gaze + IMU only, no pixels | the view settles, or on each REFIXATION |
| **1 fps + DINOv2 change** | DINOv2 CLS of every frame | cosine to the **last sent** frame < τ_frame (the labels' own "frame changed" threshold) |
| **1 fps + pixel change** | 64×64 grey thumbnail of every frame | mean absolute difference to the **last sent** frame > the median consecutive-frame difference (same median rule as τ_frame) |
| **1 fps: all frames** | nothing | always (the ceiling) |

The tables below show, before any VLM run: frames sent, coverage of the questions'
moments, and what one send/skip **decision costs** in FLOPs. The DINOv2 baseline uses the
same ViT-S/14 at 98 px that made the labels. The pixel and DINOv2 baselines also need the
camera to capture and process every frame; the gaze/IMU gate does not.

A sweep of each baseline's threshold is plotted against the fixed intervals, so methods can
be compared across frame budgets, not only at their default threshold.

In [ ]:
from vlmeval import baselines as bl
from eyehead import policy as pol

TAU_F = json.load(open(os.path.join(DRIVE_DIR, 'thresholds.json')))['tau_frame']
cls_by_clip = pol.load_cls(FEAT_CACHE, clips)            # L2-normalised CLS, aligned with each clip
assert all(F is not None for F in cls_by_clip), 'feat_cache.npz does not cover these videos'
thumbs = {c['sequence']: bl.thumbnails(metas[c['sequence']], c['frame_no']) for c in clips}
PIX_THR = float(np.median(np.concatenate([bl.consecutive_pixel_diffs(t) for t in thumbs.values()])))
print(f'tau_frame (DINOv2 CLS cosine) = {TAU_F:.4f}   pixel threshold (median grey MAD) = {PIX_THR:.2f}')

ALL, PIX, DINO = '1 fps: all frames', '1 fps + pixel change', '1 fps + DINOv2 change'
def add_baselines(sel, pix_thr=PIX_THR, tau=TAU_F, names=(ALL, PIX, DINO)):
    out = {n: {} for n in names}
    for c, F in zip(clips, cls_by_clip):
        v, fn = c['sequence'], c['frame_no']
        out[names[0]][v] = [int(f) for f in fn]
        out[names[1]][v] = bl.pixel_select(fn, thumbs[v], pix_thr)
        out[names[2]][v] = bl.dino_select(fn, F, tau)
    sel.update(out)
    return out
new = add_baselines(sel)

cov = pd.concat([cov[~cov.condition.isin(new)], egoev.coverage(qs, new, frame_time)], ignore_index=True)
ctab = cov.groupby('condition', sort=False).agg(
    questions=('qid', 'size'), median_dist_s=('dist', 'median'),
    **{f'within_{t}': (f'hit_{t}', 'mean') for t in ('1s', '2s', '5s')})
for c_ in ('within_1s', 'within_2s', 'within_5s'):
    ctab[c_] *= 100

ref = metas[videos[0]]
DEC = {'D: full': bl.gate_decision_flops(), DINO: bl.vit_flops(98),
       PIX: bl.pixel_decision_flops(ref['orig_w'], ref['orig_h']), ALL: 0}
secs = sum(n_frames.values())
cmp_rows = []
for m in ['D: full', DINO, PIX, ALL, 'fixed every 5 s', 'fixed every 6 s']:
    fr = sum(len(x) for x in sel[m].values())
    cmp_rows.append(dict(method=m, frames_per_video=fr / len(videos), frames_per_min=60 * fr / secs,
                         within_2s=ctab.loc[m, 'within_2s'],
                         decision_flops_per_frame=DEC.get(m, 0),
                         decision_GFLOPs_per_min=60 * DEC.get(m, 0) / 1e9))
cmp0 = pd.DataFrame(cmp_rows).set_index('method')
print('before any VLM run, on the 12 videos:')
display(cmp0.style.format({'frames_per_video': '{:.1f}', 'frames_per_min': '{:.1f}', 'within_2s': '{:.1f}',
                           'decision_flops_per_frame': '{:,.0f}', 'decision_GFLOPs_per_min': '{:.4f}'}))
print(f"one decision: gaze/IMU gate {DEC['D: full']:,} FLOPs vs DINOv2 {DEC[DINO]/1e9:.2f} GFLOPs "
      f"({DEC[DINO]/DEC['D: full']:,.0f}x) vs pixel {DEC[PIX]/1e6:.1f} MFLOPs")

# threshold sweeps: frames sent vs coverage, against the fixed intervals
sweep = []
for q_ in (20, 30, 40, 50, 60, 70, 80, 90):
    t_ = float(np.percentile(np.concatenate([bl.consecutive_pixel_diffs(t) for t in thumbs.values()]), q_))
    s_ = add_baselines({}, pix_thr=t_, names=('_a', '_p', '_d'))['_p']
    sweep.append(('pixel change', t_, s_))
for tau_ in (0.70, 0.75, 0.80, 0.85, 0.90, 0.93, 0.95):
    s_ = add_baselines({}, tau=tau_, names=('_a', '_p', '_d'))['_d']
    sweep.append(('DINOv2 change', tau_, s_))
for e in (3, 4, 5, 6, 8):
    sweep.append(('fixed interval', e, sel[f'fixed every {e} s']))
sweep.append(('Full (policy D)', 0, sel['D: full']))
rows = []
for kind, thr, s_ in sweep:
    cv = egoev.coverage(qs, {'x': s_}, frame_time)
    rows.append(dict(kind=kind, threshold=thr, frames_per_min=60 * sum(len(x) for x in s_.values()) / secs,
                     within_2s=100 * cv['hit_2s'].mean()))
sw = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8, 5))
for kind, col, mk in (('fixed interval', '0.5', 'o-'), ('pixel change', 'tab:orange', 's-'),
                      ('DINOv2 change', 'tab:green', '^-'), ('Full (policy D)', 'tab:blue', '*')):
    d_ = sw[sw.kind == kind].sort_values('frames_per_min')
    ax.plot(d_.frames_per_min, d_.within_2s, mk, color=col, ms=9 if mk == '*' else 5, label=kind)
ax.set_xlabel('frames sent per minute'); ax.set_ylabel('% of questions with a frame within 2 s of their moment')
ax.set_title('coverage vs frame budget (threshold sweeps)'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## 6d. Spatial screen: does gaze put the fine patches on the target? (no GPU)

The **spatial module** (repo `apt-vlm`, `src/models/foveation.py`) gives a sent frame
14 px patches near the gaze point, 28 px further out and 56 px in the periphery (rings set
by the fall-off `a`). Combined with the **temporal module** (Full + policy D), the VLM gets
`frames sent x tokens per frame`. This section checks, before any VLM run, whether the
combination puts detail where the questions need it.

Gaze: the fold CSVs' `gaze_xy_window`, already projected into the upright frame with the
factory calibration (no VRS download). Target: EgoEverything's box at the keyframe, same
1408 px frame. The layout is computed exactly as the spatial module does (336 px frame,
14/28/56 px, 110 deg pinhole).

**A. Spatial module alone (keyframe).** On the frame nearest each keyframe, where does the
fine region fall, for three fovea positions:

| source | fovea at | tests |
|---|---|---|
| gaze | the wearer's gaze at that frame | the method |
| centre | the image centre | is gaze better than head direction alone? |
| gaze, other time | the same wearer's gaze at a random other moment (>= 30 s away) | is it *this* gaze, or just where this wearer usually looks? |

`ratio` = tokens the layout spends on the target / tokens a **uniform resize to the same
total** would spend: above 1 means foveation beats uniform resizing at equal cost.

**B. Temporal + spatial.** For each frame policy: the sent frame nearest the keyframe must
be within `TOL` s **and** have the target in its fovea (gaze at that sent frame).

**C. Is the gaze point reliable when each policy sends?** Gaze spread within +-0.2 s of
every sent frame: small = fixating (the fovea is where the eyes rest), large = mid-saccade.
The claim to test: D sends at settles, so its frames are fixations more often than a fixed
interval's.

**Caveats.** EgoEverything questions are built from objects the wearer looked at, so gaze
near the target is partly by construction; the *centre* and *other time* rows show how much
of the effect is real. The box is valid at the keyframe only (hence small `TOL`). Eccentricity
uses a pinhole camera; Aria RGB is fisheye (Step 4 not built).

**Go / no-go for a VLM run:** gaze clearly above both controls on target-in-fovea, and
`ratio` well above 1; for the combination, D at least level with the fixed interval on B and
above it on C.

In [ ]:
from vlmeval import spatial as sp

A_VALUES = (10, 20, 40, 80, 160)     # fall-off a: small = small fovea, few tokens
A_MAIN = 40                          # the spatial module's default
TOL = 1.0                            # B: sent frame must be within this many s of the keyframe
STABLE_DEG = 3.0                     # C: gaze spread below this = fixating

print('foveated layout: 336 px frame, 14/28/56 px patches, 110 deg pinhole (uniform = 576 tokens)')
display(sp.ring_table(A_VALUES).round(1))

tracks, n_drop = sp.gaze_tracks(FOLDS, videos, frame_time)
print(f'gaze tracks: {len(tracks)} videos, {sum(len(t) for t, _ in tracks.values()):,} samples '
      f'({n_drop} windows dropped for out-of-FOV fillers)')

def video_ci(d):
    lo, hi = egoev.video_bootstrap(d, lambda t: t['diff'].mean())
    return d['diff'].mean(), lo, hi

# ---- A. spatial module alone, on the keyframe ------------------------------------------
kf = sp.keyframe_screen(qs, tracks, metas, A_VALUES)
g = kf[(kf.source == 'gaze') & (kf.a == A_MAIN)]
chk = g[['dist2gaze', 'gaze_box_centre_px']].apply(pd.to_numeric, errors='coerce').dropna()
print(f"\nprojection check: Spearman(our gaze-to-box distance, dataset AveDist2Gaze) = "
      f"{chk.corr(method='spearman').iloc[0, 1]:.2f} on {len(chk)} questions (should be clearly > 0)")

kt = (kf[kf.a == A_MAIN].groupby('source', sort=False)
        .agg(questions=('qid', 'nunique'), gaze_inside_box=('inside', 'mean'),
             median_deg_to_box=('d_deg', 'median'), target_in_fovea=('in_fovea', 'mean'),
             tokens_per_frame=('tokens', 'mean'), tokens_on_target=('tok_box', 'mean'),
             uniform_same_total=('uni_box', 'mean'), median_ratio=('ratio', 'median')))
kt[['gaze_inside_box', 'target_in_fovea']] *= 100
print(f'\nA. on the keyframe, a = {A_MAIN} (percentages of questions):')
display(kt.round(2))

w = kf[kf.a == A_MAIN].pivot_table(index=['qid', 'video'], columns='source',
                                   values=['in_fovea', 'ratio']).dropna()
for ctrl in ('centre', 'gaze, other time'):
    for col, scale, unit in (('in_fovea', 100, 'points'), ('ratio', 1, '')):
        d = pd.DataFrame({'video': w.index.get_level_values('video'),
                          'diff': scale * (w[(col, 'gaze')] - w[(col, ctrl)]).to_numpy()})
        m_, lo, hi = video_ci(d)
        print(f'   gaze - {ctrl:<17} {col:<9} {m_:+6.2f} {unit:<6} 95% CI [{lo:+.2f}, {hi:+.2f}]  (videos resampled)')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
sw = kf.groupby(['source', 'a']).agg(tokens=('tokens', 'mean'), tok_box=('tok_box', 'mean'),
                                     in_fovea=('in_fovea', 'mean')).reset_index()
frac = (kf.uni_box / kf.tokens).mean()                     # target's share of the frame area
for src, d in sw.groupby('source', sort=False):
    ax[0].plot(d.tokens, d.tok_box, 'o-', label=src)
    ax[1].plot(d.tokens, 100 * d.in_fovea, 'o-', label=src)
n = np.linspace(sw.tokens.min(), 576, 50)
ax[0].plot(n, frac * n, 'k--', label='uniform resize')
ax[0].set(xlabel='tokens per frame', ylabel='tokens on the target', title='detail on the target')
ax[1].set(xlabel='tokens per frame', ylabel='% targets in the 14 px region', title='target in fovea')
for a_ in ax:
    a_.legend(fontsize=8); a_.grid(alpha=.3)
plt.tight_layout(); plt.show()

ex = g.sample(min(6, len(g)), random_state=0)
fig, axs = plt.subplots(1, len(ex), figsize=(3.4 * len(ex), 3.8))
for a_, r in zip(np.atleast_1d(axs), ex.itertuples()):
    m = metas[r.video]
    q = qs.set_index('qid').loc[r.qid]
    img = plt.imread(frame_path(m, r.frame)); H, W_ = img.shape[:2]
    gu, gv = sp.gaze_at(tracks[r.video], m['ts'][r.frame])
    yy, xx = np.mgrid[2:H:4, 2:W_:4]
    E = sp.ecc_deg(xx, yy, gu * W_, gv * H, img=W_)
    a_.imshow(img)
    a_.contour(xx, yy, E, levels=sp.ring_radii_deg(sp.FOV / 2, A_MAIN), colors=['red', 'orange'], linewidths=1)
    s = W_ / m['orig_w']
    x1, y1, x2, y2 = [b * s for b in q.bbox]
    a_.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='yellow', lw=2))
    a_.plot(gu * W_, gv * H, 'r+', ms=12, mew=2)
    a_.set_title(f"{q.object}\nin fovea: {bool(r.in_fovea)}, ratio {r.ratio:.1f}", fontsize=8); a_.axis('off')
plt.suptitle('gaze (+), 14 px region (red), 28 px region (orange), target box (yellow)', fontsize=9)
plt.tight_layout(); plt.show()

# ---- B. temporal + spatial ---------------------------------------------------------------
COND_TS = [c for c in ['D: full', 'D: ground truth', 'fixed every 3 s', 'fixed every 5 s', 'fixed every 6 s',
                       '1 fps + pixel change', '1 fps + DINOv2 change', '1 fps: all frames'] if c in sel]
cb = sp.combined_screen(qs, sel, COND_TS, tracks, metas, frame_time, a=A_MAIN, tol=TOL)
secs = sum(n_frames.values())
rows = []
for c in COND_TS:
    gz, ce = (cb[(cb.condition == c) & (cb.source == s_)] for s_ in ('gaze', 'centre'))
    rows.append({'condition': c, 'frames/min': 60 * sum(len(f) for f in sel[c].values()) / secs,
                 f'frame within {TOL:g} s': 100 * gz.hit_time.mean(),
                 'frame + target in fovea (gaze)': 100 * gz.hit_fovea.mean(),
                 'frame + target in fovea (centre)': 100 * ce.hit_fovea.mean(),
                 'fovea on target given frame (gaze)': 100 * gz[gz.hit_time].hit_fovea.mean(),
                 'fovea on target given frame (centre)': 100 * ce[ce.hit_time].hit_fovea.mean()})
bt = pd.DataFrame(rows).set_index('condition')
print(f'\nB. temporal + spatial, a = {A_MAIN}, TOL = {TOL:g} s (% of questions):')
display(bt.round(1))

def cond_diff(a, b, col='hit_fovea', src='gaze'):
    x = cb[(cb.condition == a) & (cb.source == src)].set_index('qid')
    y = cb[(cb.condition == b) & (cb.source == src)].set_index('qid').loc[x.index]
    return video_ci(pd.DataFrame({'video': x['video'], 'diff': 100 * (x[col].astype(float) - y[col].astype(float))}))
for b in [c for c in COND_TS if c.startswith('fixed')]:
    m_, lo, hi = cond_diff('D: full', b)
    print(f'   D: full - {b:<16} frame + target in fovea: {m_:+5.1f} points, 95% CI [{lo:+.1f}, {hi:+.1f}]')

# ---- C. is the gaze point a fixation when each policy sends? -----------------------------
st = sp.sent_frame_stability(sel, COND_TS, tracks, frame_time)
ct = st.groupby('condition', sort=False).agg(
    frames=('spread', 'size'), median_spread_deg=('spread', 'median'),
    fixating=('spread', lambda s_: 100 * (s_.dropna() < STABLE_DEG).mean()))
ct = ct.rename(columns={'fixating': f'% fixating (< {STABLE_DEG:g} deg)'})
print(f'\nC. gaze spread within +-0.2 s of each sent frame:')
display(ct.round(2))

def stab_diff(a, b):
    d = st[st.condition.isin([a, b])].dropna(subset=['spread']).copy()
    d['fix'] = (d.spread < STABLE_DEG).astype(float)
    lo, hi = egoev.video_bootstrap(d, lambda t: 100 * (t[t.condition == a].fix.mean() - t[t.condition == b].fix.mean()))
    return 100 * (d[d.condition == a].fix.mean() - d[d.condition == b].fix.mean()), lo, hi
for b in [c for c in COND_TS if c.startswith('fixed') or c == '1 fps: all frames']:
    m_, lo, hi = stab_diff('D: full', b)
    print(f'   D: full - {b:<17} % fixating: {m_:+5.1f} points, 95% CI [{lo:+.1f}, {hi:+.1f}]')

kf.to_csv(os.path.join(VLM_DIR, 'spatial_keyframe_screen.csv'), index=False)
cb.to_csv(os.path.join(VLM_DIR, 'spatial_combined_screen.csv'), index=False)
print('\nsaved per-question tables to', VLM_DIR)

## 7. Load Qwen2.5-VL-3B and smoke test

Same model and resolution as the pilot (`448*448`, about 256 visual tokens per frame).

In [ ]:
from vlmeval import qwen_mcq
import time

MODEL_ID, MAX_PIXELS = 'Qwen/Qwen2.5-VL-3B-Instruct', 448 * 448
model, proc, letter_ids, dtype = qwen_mcq.load_model(MODEL_ID)
print(MODEL_ID, dtype, f'{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU')

q = qs.iloc[0]
frames = [(frame_path(metas[q.video], k), metas[q.video]['ts'][k])
          for k in sel['hybrid 50/50 (D budget)'][q.video]]
t0 = time.time()
p, ntok = qwen_mcq.score(model, proc, letter_ids,
                         qwen_mcq.build_messages(q.question, q.options, frames, MAX_PIXELS))
print(f'{len(frames)} frames, {ntok} prompt tokens, {time.time()-t0:.1f} s')
print('Q:', q.question)
for L, o, pi in zip('ABCDE', q.options, p):
    print(f"   {L}. {o:<60} p={pi:.3f}{'   <- answer' if L == q.answer else ''}")
assert np.isfinite(p).all(), 'NaN logits: use an L4/A100 (bf16) or lower MAX_PIXELS'

## 8. Score Full against the 1 fps baselines

All on the 834 questions of the 12 videos (D: full is already scored and reused):

| condition | frames per question (12-video average) | rough time on an A100 |
|---|---|---|
| D: full | ~28 | done |
| 1 fps + DINOv2 change | see 6c | ~1.5–2.5 s per question |
| 1 fps + pixel change | see 6c | ~1.5–2.5 s per question |
| 1 fps: all frames | ~146 (up to 299) | **~10–20 s per question, several hours** |

All frames runs last, so the others finish first; stop it whenever you like (or set
`QUICK = True` for the 4 test videos). If a prompt runs out of GPU memory the frame size is
halved for that question and recorded in `max_pixels`.

In [ ]:
# QUICK = True: only the 4 TEST videos (204 questions); False: all 12 videos (834 questions)
QUICK = False
CONDITIONS = ['D: full', DINO, PIX, ALL]          # all frames last: by far the slowest
# also available: fixed every 3/4/5/6/8 s, D + max gap 5/6/8/10 s: full, uniform (D budget), ...
missing = [c for c in CONDITIONS if c not in sel]
assert not missing, f'unknown condition names {missing}; available: {list(sel)}'

RUN_VIDEOS = sorted(set(videos) & test_videos) if QUICK else videos
qs_run = qs[qs.video.isin(RUN_VIDEOS)].reset_index(drop=True)
print(f"{len(qs_run)} questions on {len(RUN_VIDEOS)} videos x {len(CONDITIONS)} conditions")
print(pd.DataFrame({c: {v: len(sel[c][v]) for v in RUN_VIDEOS} for c in CONDITIONS})
      .assign(video_s=[n_frames[v] for v in RUN_VIDEOS]).rename_axis("frames sent per video"))

OUT = os.path.join(VLM_DIR, f"{MODEL_ID.split('/')[-1]}_px{int(MAX_PIXELS**0.5)}_all12.jsonl")
qwen_mcq.run(model, proc, letter_ids, qs_run, sel, metas, OUT, CONDITIONS, MAX_PIXELS, tag=MODEL_ID)

## 9. Results

- **Accuracy per condition**, 95% interval from resampling the 12 videos.
- **Paired comparisons** on the same questions (`diff` = a − b in accuracy points;
  `a_only_right` / `b_only_right` = questions only one condition got right; `p_mcnemar`).
- **Accuracy by target coverage** (a frame within 2 s of the keyframe sent or not).
- **Per video**, so one video with many questions cannot drive the result unseen, and
  **test vs held-out training videos** separately.

In [ ]:
res = egoev.load_results(OUT)
bad = res[res['error'].notna()] if 'error' in res else res.iloc[:0]
if len(bad):
    print(f'{len(bad)} questions not scored (oom/nan), left out below:')
    display(bad.groupby(['condition', 'error']).size())
res = res[res['error'].isna()].copy() if 'error' in res else res.copy()
res['correct'] = res['correct'].astype(int)
res = res[res.video.isin(RUN_VIDEOS)]                # only the videos of this run (section 8)
res['split'] = np.where(res.video.isin(test_videos), 'test', 'train (held out)')

acc = egoev.accuracy_table(res)
display(acc.round(2))

# the comparison table: accuracy, frames, visual tokens, and compute
vc = bl.vlm_costs(model, MAX_PIXELS)
run_secs = sum(n_frames[v] for v in RUN_VIDEOS)
crow = []
for m in [x for x in ['D: full', DINO, PIX, ALL, 'fixed every 5 s', 'fixed every 6 s'] if x in acc.index]:
    fr = sum(len(sel[m][v]) for v in RUN_VIDEOS)
    fq = acc.loc[m, 'frames_per_q']
    crow.append({'method': m, 'accuracy %': acc.loc[m, 'accuracy'],
                 '95% CI': f"[{acc.loc[m, 'ci_low']:.1f}, {acc.loc[m, 'ci_high']:.1f}]",
                 'frames sent / video': fr / len(RUN_VIDEOS), 'frames / min': 60 * fr / run_secs,
                 'visual tokens / question': fq * vc['tokens_per_frame'],
                 'decision GFLOPs / min': 60 * DEC.get(m, 0) / 1e9,
                 'VLM vision TFLOPs / question': fq * vc['vision_flops_per_frame'] / 1e12,
                 'VLM prefill TFLOPs / question': fq * vc['llm_prefill_flops_per_frame'] / 1e12})
cost = pd.DataFrame(crow).set_index('method')
print(f"VLM: {vc['tokens_per_frame']} visual tokens per frame at {int(MAX_PIXELS**0.5)} px; "
      f"vision encoder {vc['vision_params']/1e6:.0f}M params, LLM {vc['llm_params']/1e9:.2f}B params "
      f"(FLOPs ignore attention, so they are lower bounds)")
display(cost.style.format({'accuracy %': '{:.1f}', 'frames sent / video': '{:.1f}', 'frames / min': '{:.1f}',
                           'visual tokens / question': '{:,.0f}', 'decision GFLOPs / min': '{:.4f}',
                           'VLM vision TFLOPs / question': '{:.1f}', 'VLM prefill TFLOPs / question': '{:.1f}'}))

pairs = [('D: full', DINO), ('D: full', PIX), ('D: full', ALL), (DINO, ALL), (PIX, ALL)] + \
        [(f'D + max gap {g} s: full', f'fixed, rate of D + max gap {g} s') for g in GAPS] + \
        [(f'D + max gap {g} s: full', 'D: full') for g in GAPS] + \
        [('D: full', f'fixed every {e} s') for e in (3, 4, 5, 6, 8)] + [
         ('D: full', 'uniform (D budget)'), ('hybrid 50/50 (D budget)', 'uniform (D budget)'),
         ('E: full', 'uniform (E budget)'), ('D: ground truth', 'fixed every 5 s'),
         ('fixed every 5 s', 'blind (no frames)')]
have = set(res.condition)
pt = pd.DataFrame([egoev.paired_test(res, a, b) for a, b in pairs if a in have and b in have])
print('paired, same questions (diff = a - b, accuracy points):')
display(pt.round(3))

m = res.merge(cov[['condition', 'qid', 'hit_2s']], on=['condition', 'qid'], how='left')
by_cov = m[m.condition != 'blind (no frames)'].groupby(['condition', 'hit_2s']).correct.agg(['mean', 'size'])
by_cov['mean'] *= 100
print('accuracy (%) when a frame within 2 s of the keyframe was / was not sent:')
display(by_cov.round(1).unstack())

print('accuracy (%) per video:')
pv = res.pivot_table(index='video', columns='condition', values='correct', aggfunc='mean') * 100
pv.insert(0, 'questions', res.groupby('video').qid.nunique())
display(pv.round(1))
print('accuracy (%) on test vs held-out training videos:')
display((res.pivot_table(index='split', columns='condition', values='correct', aggfunc='mean') * 100).round(1))

fig, ax = plt.subplots(figsize=(9, 5))
order = acc.sort_values('accuracy').index
ax.barh(order, acc.loc[order, 'accuracy'],
        xerr=[acc.loc[order, 'accuracy'] - acc.loc[order, 'ci_low'],
              acc.loc[order, 'ci_high'] - acc.loc[order, 'accuracy']], color='tab:blue', alpha=.8)
ax.axvline(20, ls=':', c='k', lw=1, label='chance (20%)')
ax.set_xlabel('accuracy (%)'); ax.set_title(f'{MODEL_ID}, {len(qs)} questions, {len(videos)} videos')
ax.legend(); plt.tight_layout(); plt.show()

# accuracy against frame rate: the fixed intervals trace what a fixed interval buys at each
# rate; D is a point. D beats fixed-interval sampling only if it sits above that line.
bt = egoev.budget_table(sel, n_frames)
pts = acc.join(bt['sends_per_min'], how='left')
fx = pts[pts.index.str.startswith('fixed')].sort_values('sends_per_min')
fig, ax = plt.subplots(figsize=(8, 5))
if len(fx):
    ax.errorbar(fx['sends_per_min'], fx['accuracy'],
                yerr=[fx['accuracy'] - fx['ci_low'], fx['ci_high'] - fx['accuracy']],
                fmt='-o', color='0.5', capsize=3, label='fixed interval (one interval for all videos)')
    for name, r in fx.iterrows():
        ax.annotate(name.replace('fixed every ', '').replace('fixed, rate of ', '='), (r['sends_per_min'], r['accuracy']),
                    xytext=(4, 4), textcoords='offset points', fontsize=8, color='0.4')
for name, col in [('D: full', 'tab:blue'), ('uniform (D budget)', 'tab:orange'),
                  ('hybrid 50/50 (D budget)', 'tab:purple'), ('E: full', 'tab:green')] + \
                 [(f'D + max gap {g} s: full', 'tab:red') for g in GAPS] + \
                 [(DINO, 'tab:olive'), (PIX, 'tab:brown'), (ALL, 'black')]:
    if name in pts.index:
        r = pts.loc[name]
        ax.errorbar([r['sends_per_min']], [r['accuracy']],
                    yerr=[[r['accuracy'] - r['ci_low']], [r['ci_high'] - r['accuracy']]],
                    fmt='s', color=col, capsize=3, label=name)
ax.set_xlabel('frames sent per minute of video'); ax.set_ylabel('accuracy (%)')
ax.set_title('accuracy vs frame rate: above the grey line = better than a fixed interval')
ax.grid(alpha=.3); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

---

## Reading it

| look at | meaning |
|---|---|
| **section 6b advantage_2s** | whether the gap cap sends frames near question moments more often than a fixed interval at the same cost; if not, a VLM gain is unlikely |
| **D + max gap vs fixed at the same rate** (paired) | the main result of this run |
| D + max gap vs D | what the cap itself buys |
| accuracy vs frame-rate plot | a policy beats fixed-interval sampling only if it sits above the grey line |
| per video / split | no single video or split should carry the result |

## Caveats

- Qwen2.5-VL-3B at 448 px: absolute accuracy is low; the comparison between conditions is
  what matters. Accuracy was flat between 26 and 31 frames per question, so this VLM may
  be the bottleneck rather than frame choice.
- 12 videos, one with 111 questions: intervals are wide.
- The equal-cost fixed interval's rate is set from these 12 videos' totals (a single number,
  so still deployable, but tuned to match the policy's cost).